In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [45]:
def adapt_discover_to_ace(ace_df, disc_df, all_targets, split_date="2021-01-01", scaler=True):
    """
    Адаптация discover -> ace
    1. L обучается только на пересечении тренировочных данных ace и discover
    2. Тест discover адаптируется в домен данных ace
    3. Из всех данных возвращается только discover_test_adapted для дальнейшего предсказания
    """
    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]   # адаптируем только тест discover

    # Пересечение индексов внутри train
    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in all_targets]

    # Масштабирование (если задано)
    if scaler:
        sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
        sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

        X_ace = sc_ace.transform(ace_overlap[feature_cols])
        X_disc = sc_disc.transform(disc_overlap[feature_cols])
    else:
        sc_ace = sc_disc = None
        X_ace = ace_overlap[feature_cols].values
        X_disc = disc_overlap[feature_cols].values

    # Модель адаптации L: discover -> ace
    L = LinearRegression()
    L.fit(X_disc, X_ace)

    if scaler:
        X_disc_test = sc_disc.transform(disc_test[feature_cols])            # Нормировка тестового набора данных discover
        X_disc_test_adapted = L.predict(X_disc_test)                        # Применение обученной модели адаптации
        X_disc_test_adapted = sc_ace.inverse_transform(X_disc_test_adapted) # Приводим новые адаптированные данные к ненормированному виду
    else:
        X_disc_test_adapted = L.predict(disc_test[feature_cols])

    disc_test_adapted = pd.DataFrame(X_disc_test_adapted, index=disc_test.index, columns=feature_cols)

    # Целевые переменные возвращаются обратно неизменёнными
    for col in all_targets:
        disc_test_adapted[col] = disc_test[col]

    return disc_test_adapted, L, sc_disc, sc_ace

In [35]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("scaler", StandardScaler()),
        ("boost", LGBMRegressor(
        metric='mse',
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=0.1,
        reg_alpha=0.05,
        random_state=random_state,
        n_jobs=-1,
        verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(256, 128, 64),
            activation='relu',
            alpha=0.0005,
            learning_rate_init=0.001,
            solver='adam',
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1000,
            random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_M_A(ace_df, disc_test_adapted_df, split_date, all_targets, target):
    """
    Модель M_A обучается на ace_train и тестируется на адаптированном discover_test
    """
    split_date = pd.Timestamp(split_date)

    ace_train = ace_df.loc[:split_date]

    feature_cols = [c for c in ace_train.columns if c not in all_targets]

    X_train = ace_train[feature_cols].values
    y_train = ace_train[target].values

    X_test = disc_test_adapted_df[feature_cols].values
    y_test = disc_test_adapted_df[target].values

    results = {}
    results[target] = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results[target][name] = dict(mse=mse, r2=r2)

        print(f"{name}: MSE={mse:.3f}, R2={r2:.3f}")

    return results

In [32]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для адаптации
split_date = "2021-01-01"
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3']
results_all = {}

In [37]:
disc_adapted_24, L_24, sc_disc_24, sc_ace_24 = adapt_discover_to_ace(data_ace_24_copy, data_discover_24_copy, targets, split_date)

In [41]:
disc_adapted_af, L_af, sc_disc_af, sc_ace_af = adapt_discover_to_ace(data_ace_af_copy, data_discover_af_copy, targets, split_date)

In [25]:
print(f"\n==== Depth - 24h ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_24_copy, disc_adapted_24, split_date, targets, target_col)
    results_all[target_col] = res


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====
Linear: MSE=10.631, R2=0.964
Ridge: MSE=10.631, R2=0.964
Lasso: MSE=10.627, R2=0.964
LGBM: MSE=10.087, R2=0.966
MLP: MSE=10.118, R2=0.966

==== Forecast of DST_PLUS2 ====
Linear: MSE=25.406, R2=0.913
Ridge: MSE=25.406, R2=0.913
Lasso: MSE=25.401, R2=0.913
LGBM: MSE=22.644, R2=0.923
MLP: MSE=23.649, R2=0.919

==== Forecast of DST_PLUS3 ====
Linear: MSE=41.443, R2=0.859
Ridge: MSE=41.443, R2=0.859
Lasso: MSE=41.442, R2=0.859
LGBM: MSE=37.724, R2=0.871
MLP: MSE=47.080, R2=0.840


In [42]:
print(f"\n==== Depth - autocorrelation function ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_af_copy, disc_adapted_af, split_date, targets, target_col)
    results_all[target_col] = res


==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====
Linear: MSE=10.725, R2=0.963
Ridge: MSE=10.725, R2=0.963
Lasso: MSE=10.722, R2=0.963
LGBM: MSE=10.593, R2=0.964
MLP: MSE=9.947, R2=0.966

==== Forecast of DST_PLUS2 ====
Linear: MSE=25.788, R2=0.912
Ridge: MSE=25.788, R2=0.912
Lasso: MSE=25.784, R2=0.912
LGBM: MSE=23.388, R2=0.920
MLP: MSE=25.799, R2=0.912

==== Forecast of DST_PLUS3 ====
Linear: MSE=42.075, R2=0.856
Ridge: MSE=42.075, R2=0.856
Lasso: MSE=42.071, R2=0.856
LGBM: MSE=38.053, R2=0.870
MLP: MSE=48.711, R2=0.833
